In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.lines as lines
from matplotlib import collections  as mc
from collections import *
from matplotlib.pyplot import figure
from matplotlib.patches import Rectangle
import os,gzip

In [5]:
# ---------- Helper functions ----------
def getChrSize(file):
    """
    Reads a chromosome size file and returns a dict: {'LG1': length, ...}
    """
    chrSizeDict = {}
    with open(file) as inf:
        for line in inf:
            line = line.rstrip()
            chrom, _, length = line.split()
            chrSizeDict[chrom] = length
    return chrSizeDict


def getchrPos(LG1=[21.6, 22.4]):
    """
    Returns a dict of Y-axis positions for chromosomes (for plotting layout)
    Example output:
    {'LG1': [21.6, 22.4], 'LG2': [20.6, 21.4], ..., 'LG23': [-0.4, 0.4]}
    """
    chrPosList = {}
    for i in range(1, 24):
        chrPosList[f'LG{i}'] = [x - 1 * (i - 1) for x in LG1]
    return chrPosList


def getDNMPos(DNM_pos_file):
    """
    Reads inversion bed-like file and counts line occurrences.
    Returns a Counter dict: {(chr, pos1, pos2): count}
    """
    DNMPosList = []
    with open(DNM_pos_file) as inf:
        for line in inf:
            line = line.rstrip()
            DNMPosList.append(tuple(line.split()[:3]))
    return Counter(DNMPosList)


def addLines(DNMbed, newpos):
    """
    Creates inversion line segments for background plotting.
    Each line corresponds to an inversion interval.
    """
    chrPosList = getchrPos()
    DNMPos = getDNMPos(DNMbed)
    lines = []
    for k, v in DNMPos.items():
        chrom, pos1, pos2 = k
        if not chrom.startswith("LG"):
            continue
        lines.append([
            (float(pos1), float(chrPosList[chrom][0]) + newpos),
            (float(pos2), float(chrPosList[chrom][0]) + newpos)
        ])
    return lines

In [7]:
# Example data
fig = plt.figure(figsize=(10,8), dpi = 1000)

ax = fig.add_subplot()
chrSizeDict = getChrSize('~/3.SV/0.Scripts/female.guppy.genome.size')

chrNames = chrSizeDict.keys()
chroms = list(chrNames)[::-1]
chroms = [c.replace("LG", "Chr") for c in list(chrNames)[::-1]]
chr_len = [int(i) for i in chrSizeDict.values()][::-1]


hbars = ax.barh(chroms, chr_len, align='center', color = 'grey', alpha = 0.3)
ax.set_xticks([0,5000000,10000000,15000000,20000000,25000000,30000000,35000000,40000000,45000000])
labels = [0,5,10,15,20,25,30,35,40,45]
ax.set_xticklabels(labels)
ax.set_xlim(0,45000000)
#set_axis_off()
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines["bottom"].set_position(['data',-0.5])

ax.tick_params(axis='y', which=u'both',length=0)

ax.set_xlabel('Genomic Distribution (Mb)')


#### frame ####
YAHlines = addLines('~/3.SV/4.localPCA/female_genome/Yarra.final.inv', 0.15)
QUHlines = addLines('~/3.SV/4.localPCA/female_genome/Quare.final.inv', 0.40)
ARHlines = addLines('~/3.SV/4.localPCA/female_genome/Aripo.final.inv', 0.65)


lc1 = mc.LineCollection(ARHlines, colors='red', linewidths=3, alpha=0.8)
lc3 = mc.LineCollection(QUHlines, colors='green', linewidths=3, alpha=0.8)
lc5 = mc.LineCollection(YAHlines, colors='blue', linewidths=3, alpha=0.8)

ax.add_collection(lc1)
ax.add_collection(lc3)
ax.add_collection(lc5)
ax.legend(["Aripo",  "Quare", "Yarra"], title = 'Population',loc=5, ncol=1, frameon=False)

currentAxis = plt.gca()

plt.savefig('~/3.SV/plots/DNMPlot.pdf', dpi = 1000, pad_inches=0)

plt.show()

In [8]:


# ---------- Main plotting function ----------

def plotGenomeLayout(filtered_df, chrSizeFile, inversionFiles, outputFile, dpi=1000):
    """
    Plot genome layout with chromosome bars, inversion lines, and filtered points.

    Parameters:
    - filtered_df: pandas DataFrame with ['pos', 'chr2', 'source'] columns
    - chrSizeFile: path to chromosome size file
    - inversionFiles: dict of {'Label': (file_path, y_offset, color)}
    - outputFile: output path (.pdf or .png)
    - dpi: resolution
    """

    # --- Prepare chromosome size and layout ---
    chrSizeDict = getChrSize(chrSizeFile)
    chrNames = list(chrSizeDict.keys())[::-1]  # reverse for top-to-bottom
    chrNames_plot = [k.replace('LG', 'Chr') for k in list(chrSizeDict.keys())[::-1]]
    chr_len = [int(chrSizeDict[chr]) for chr in chrNames]
    chrPosList = getchrPos()

    fig, ax = plt.subplots(figsize=(10, 8), dpi=dpi)

    # Plot chromosome bars
    hbars = ax.barh(chrNames_plot, chr_len, align='center', color='grey', alpha=0.3)
    ax.set_xlim(0, max(chr_len) * 1.05)
    ax.set_xticks([0, 5e6, 10e6, 15e6, 20e6, 25e6, 30e6, 35e6, 40e6, 45e6])
    ax.set_xticklabels([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])

    # Style
    ax.spines['top'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_position(['data', -0.5])
    ax.tick_params(axis='y', length=0)
    ax.set_xlabel('Genomic Distribution (Mb)')

    # --- Add inversion lines ---
    for label, (file_path, y_offset, color) in inversionFiles.items():
        lines = addLines(file_path, y_offset)
        lc = mc.LineCollection(lines, colors=color, linewidths=3, alpha=0.8, label=label)
        ax.add_collection(lc)

    # --- Prepare filtered_df ---
    filtered_df = filtered_df.copy()
    # convert chr2 to string LG labels
    filtered_df["chr2"] = filtered_df["chr2"].astype(int)
    filtered_df["chr2"] = filtered_df["chr2"].apply(lambda x: f"LG{x}")
    filtered_df["pos"] = filtered_df["pos"].astype(float)

    source_colors = {
        "auto": "grey",
        "X": "grey",
        "Y": "grey"
    }

    # --- Plot each source as horizontal bars ---
    for src, group in filtered_df.groupby("source"):
        color = source_colors.get(src, "grey")
        for chrom, sub in group.groupby("chr2"):
            if chrom not in chrPosList:
                continue
            y_bottom = float(chrPosList[chrom][0])
            y_top = float(chrPosList[chrom][1])
            # Draw each point as a very thin horizontal rectangle across the chromosome height
            for pos in sub["pos"].astype(float):
                ax.add_patch(
                    plt.Rectangle(
                        (pos, y_bottom),  
                        width=1,          
                        height=y_top - y_bottom,  
                        color=color,
                        alpha=0.4
                    )
                )


    pie_traits = {"car_PIE", "mel_PIE"}
    
    # Assign preliminary category
    filtered_df["category"] = np.where(filtered_df["trait"].isin(pie_traits), "PIE", "others")
    
    
    # Find overlaps per chromosome and position
    overlap_mask = (
        filtered_df.groupby(["chr2", "pos"])["category"]
        .transform(lambda x: "overlap" if set(x) == {"PIE", "others"} else x.iloc[0])
    )
    
    filtered_df["category"] = overlap_mask
    
    # Map to colors
    color_map = {
        "PIE": "orange",
        "others": "grey",
        "overlap": "red"
    }
    
    # --- Plotting ---
    for (chrom, pos_group), group in filtered_df.groupby(["chr2", "pos"]):
        if chrom not in chrPosList:
            continue
        y_bottom = float(chrPosList[chrom][0])
        y_top = float(chrPosList[chrom][1])
        
        for _, row in group.iterrows():
            ax.add_patch(
                plt.Rectangle(
                    (row["pos"], y_bottom),
                    width=1,
                    height=y_top - y_bottom,
                    color=color_map[row["category"]],
                    alpha=0.4
                )
            )


    # --- Legend ---
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))


    # --- Y-axis labels as LG1–LG23 ---
    ax.set_yticks([i for i in range(len(chrNames))])
    ax.set_yticklabels(chrNames_plot)

    # --- Save and show ---
    plt.savefig(outputFile, dpi=dpi, bbox_inches='tight')
    plt.show()

In [ ]:
# the GWAS plot
# Define the directory path
path = "~/3.SV/4.localPCA/female_genome/color_gwas"
os.chdir(path)

# List all .gz files
gz_files = [f for f in os.listdir(path) if f.endswith(".gz")]

dfs = []
for file in gz_files:
    with gzip.open(file, 'rt') as f: 
        df = pd.read_csv(f, sep=",")  
        dfs.append(df)

# Combine all dataframes
combined_df = pd.concat(dfs, ignore_index=True)
filtered_df = combined_df[
    (combined_df["source"] == "auto") |
    (((combined_df["source"] == "X") | (combined_df["source"] == "Y")) & (combined_df["chr2"] == 12))
]


In [5]:
# Define inversion files
inversionFiles = {
    "Aripo": ("~/3.SV/4.localPCA/female_genome/Aripo.final.inv", 0.65, "red"),
    "Quare": ("~/3.SV/4.localPCA/female_genome/Quare.final.inv", 0.40, "green"),
    "Yarra": ("~/3.SV/4.localPCA/female_genome/Yarra.final.inv", 0.15, "blue")
}


plotGenomeLayout(
    filtered_df,
    "~/3.SV/0.Scripts/female.guppy.genome.size",
    inversionFiles,
    "~/3.SV/plots/DNMPlot_combined.pdf"
)
